# Turkish chunk prep for POWSM LoRA

**Local:** set the notebook’s working directory to `sig/fine-tune` or `sig/fine-tune/notebooks` (or `cd` there in the terminal before `jupyter`), so the first cell can import `turkish_lora_util.py`.

Reads WAV + TextGrid from `sig/fine-tune/data/task-1` and `task-2`, writes 20 s / 16 kHz chunks to `sig/fine-tune/data/turkish_chunks/`.

- **Phone tiers:** task-1 → `phones`, task-2 → `REF-phones`.
- **Chunking:** phones whose interval **midpoint** lies in `[t, t+20s)` are included (avoids dropping boundary spans).
- Built-in preprocessing drops obvious annotation noise and normalizes common phone variants before chunk writing.
- Optional **`phone_map.json`** aliases override the built-in rules (run the vocab cell after prep, edit aliases, then re-run prep if needed).
- Each prep run also writes `phone_audit.json` so you can inspect mapped, dropped, and still-unresolved phones.

In [1]:
from __future__ import annotations

import json
import sys
from pathlib import Path

_here = Path.cwd().resolve()
if (_here / "turkish_lora_util.py").is_file():
    fine_tune_root = _here
elif (_here.parent / "turkish_lora_util.py").is_file():
    fine_tune_root = _here.parent
else:
    raise SystemExit(
        "Run this notebook with cwd = sig/fine-tune or sig/fine-tune/notebooks"
    )
sys.path.insert(0, str(fine_tune_root))

from turkish_lora_util import (
    DEFAULT_PHONE_ALIASES,
    DEFAULT_PHONE_AUDIT,
    DEFAULT_CHUNKS_DIR,
    DEFAULT_RAW_DATA,
    PHONE_TIER_BY_TASK,
    chunk_corpus,
    load_phone_map,
    write_phone_audit,
    write_splits,
)

DATA_DIRS = {
    "task1": DEFAULT_RAW_DATA / "task-1",
    "task2": DEFAULT_RAW_DATA / "task-2",
}
OUT_DIR = DEFAULT_CHUNKS_DIR
PHONE_AUDIT_PATH = DEFAULT_PHONE_AUDIT
PHONE_MAP_PATH = OUT_DIR / "phone_map.json"

print("PHONE_TIER_BY_TASK:", PHONE_TIER_BY_TASK)
for k, p in DATA_DIRS.items():
    print(k, "exists" if p.is_dir() else "MISSING", p)

PHONE_TIER_BY_TASK: {'task1': 'phones', 'task2': 'REF-phones'}
task1 exists C:\Users\faruq\Desktop\college\senior\sig\fine-tune\data\task-1
task2 exists C:\Users\faruq\Desktop\college\senior\sig\fine-tune\data\task-2


In [2]:
# Optional: print TextGrid tier names (spot-check; no strict TextGrid parser)
import re


def _read_tg_unicode(p: Path) -> str:
    b = p.read_bytes()
    if b.startswith(b"\xff\xfe") or b.startswith(b"\xfe\xff"):
        return b.decode("utf-16", errors="replace")
    return b.decode("utf-8", errors="replace")


for task, d in DATA_DIRS.items():
    sample = next(iter(sorted(d.glob("*.TextGrid"))), None)
    if not sample:
        continue
    raw = _read_tg_unicode(sample)
    names = re.findall(r'name = "([^"]+)"', raw)
    print(task, sample.name, names)

task1 S10T1.TextGrid ['words', 'REF-words', 'phones', 'C/V', 'lexicalStress', 'linking', 'intonation']
task2 S10T2.TextGrid ['SPEAKER', 'REF', 'REF-words-matched', 'REF-phones', 'C/V', 'lexicalStress', 'intonation_type', 'intonation_accu', 'linking1', 'linking2']


In [3]:
pmap = load_phone_map(PHONE_MAP_PATH)
manifest, audit = chunk_corpus(
    DATA_DIRS,
    OUT_DIR,
    phone_map=pmap or None,
    return_audit=True,
)
audit_payload = write_phone_audit(PHONE_AUDIT_PATH, audit)

(OUT_DIR / "manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
write_splits(OUT_DIR, manifest)

train = json.loads((OUT_DIR / "train.json").read_text(encoding="utf-8"))
val = json.loads((OUT_DIR / "val.json").read_text(encoding="utf-8"))
test = json.loads((OUT_DIR / "test.json").read_text(encoding="utf-8"))
print("total chunks", len(manifest))
print("train/val/test", len(train), len(val), len(test))
print("chunks dropped after normalization", audit_payload["chunks_dropped_empty"])
print("top mapped phones", list(audit_payload["mapped_phone_counts"].items())[:10])
print("top dropped phones", list(audit_payload["dropped_phone_counts"].items())[:10])
print("wrote", PHONE_AUDIT_PATH)

total chunks 1156
train/val/test 998 75 83
chunks dropped after normalization 11
top mapped phones [('ej', 2825), ('ow', 1455), ('ɚ', 1445), ('aj', 1252), ('tʃ', 1207), ('ç', 852), ('aw', 599), ('ɝ', 530), ('dʒ', 452), ('ɔj', 138)]
top dropped phones [('spn', 1898), ('rd', 4), ('nsıd', 1), ('ᴊ', 1), ('nt', 1), ('ɜː d', 1), ('zing', 1), ('st', 1), ('rt', 1), ('rk', 1)]
wrote C:\Users\faruq\Desktop\college\senior\sig\fine-tune\data\turkish_chunks\phone_audit.json


In [4]:
# Verify WAV length (320_000 samples @ 16 kHz × 20 s)
import soundfile as sf

bad = []
for c in manifest[:50]:
    a, r = sf.read(OUT_DIR / f"{c['id']}.wav")
    if r != 16000 or a.shape[0] != 320_000:
        bad.append((c["id"], r, a.shape[0]))
print("sample check (first 50):", "ok" if not bad else bad)

sample check (first 50): ok


## POWSM vocabulary audit → `phone_map.json`

Requires `espnet` + `espnet_model_zoo`. The prep step already applies built-in normalization; use this section to inspect what is still outside POWSM and add manual overrides only where needed.

Fill `aliases` so every remaining problematic phone maps to a token **without** slashes (same string POWSM uses inside `/.../`). Re-run the prep cell after editing the map if aliases change.

In [5]:
from collections import Counter

from espnet2.bin.s2t_inference import Speech2Text

s2t = Speech2Text.from_pretrained(
    "espnet/powsm",
    device="cpu",
    lang_sym="<unk>",
    task_sym="<pr>",
)
vocab = set(s2t.converter.token2id.keys())
powsm_phones = {t.strip("/") for t in vocab if t.startswith("/") and t.endswith("/")}

tg_phones = {p for c in manifest for p in c["phones"]}
unknown = sorted(tg_phones - powsm_phones)
unknown_counts = Counter(
    p
    for c in manifest
    for p in c["phones"]
    if p in unknown
)
print("Unique phones in manifest:", len(tg_phones))
print("Unknown vs POWSM slash-tokens after normalization:", len(unknown))
print("Top unresolved phones:", unknown_counts.most_common(20))
print(unknown[:40], "..." if len(unknown) > 40 else "")

prev: dict = {}
if PHONE_MAP_PATH.is_file():
    prev = json.loads(PHONE_MAP_PATH.read_text(encoding="utf-8"))
audit_payload = {}
if PHONE_AUDIT_PATH.is_file():
    audit_payload = json.loads(PHONE_AUDIT_PATH.read_text(encoding="utf-8"))

payload = {
    "description": "Override built-in normalization for TextGrid phone labels. Values are POWSM phone tokens without slashes; empty string drops the phone.",
    "aliases": prev.get("aliases", {}),
    "default_aliases": DEFAULT_PHONE_ALIASES,
    "unique_textgrid_phones": sorted(audit_payload.get("raw_phone_counts", {}).keys())
    or prev.get("unique_textgrid_phones")
    or sorted(tg_phones),
    "unknown_after_normalization": unknown,
    "top_unknown_after_normalization": unknown_counts.most_common(30),
    "mapped_by_normalizer": audit_payload.get("mapped_phone_counts", {}),
    "dropped_by_normalizer": audit_payload.get("dropped_phone_counts", {}),
}
PHONE_MAP_PATH.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding="utf-8")
print("Wrote", PHONE_MAP_PATH)

Failed to import Flash Attention, using ESPnet default: No module named 'flash_attn'


Fetching 12 files:   0%|          | 0/12 [00:00<?, ?it/s]

Unique phones in manifest: 81
Unknown vs POWSM slash-tokens after normalization: 0
Top unresolved phones: []
[] 
Wrote C:\Users\faruq\Desktop\college\senior\sig\fine-tune\data\turkish_chunks\phone_map.json


In [6]:
unknown

[]

In [13]:
import json
from pathlib import Path
from collections import Counter

base = Path("../data/turkish_chunks")  # veya 01'in yazdığı gerçek klasör
test_items = json.loads((base / "test.json").read_text(encoding="utf-8"))
cnt = Counter(p for x in test_items for p in x["phones"])
print("spn =", cnt["spn"])
print("ej  =", cnt["ej"])
print(cnt.most_common(20))


spn = 0
ej  = 0
[('ə', 476), ('ɪ', 454), ('n', 432), ('t', 410), ('ɹ', 341), ('s', 334), ('j', 304), ('w', 276), ('d', 206), ('æ', 196), ('z', 193), ('ɛ', 174), ('e', 170), ('ʃ', 165), ('ɐ', 153), ('iː', 144), ('m', 138), ('k', 116), ('f', 113), ('i', 109)]


In [7]:
tg_phones

{'a',
 'aː',
 'b',
 'bʲ',
 'c',
 'cʰ',
 'cʷ',
 'd',
 'dʲ',
 'd̪',
 'e',
 'f',
 'fʲ',
 'h',
 'i',
 'iː',
 'j',
 'k',
 'kʰ',
 'l',
 'm',
 'mʲ',
 'm̩',
 'n',
 'n̩',
 'o',
 'p',
 'pʰ',
 'pʲ',
 'q',
 'r',
 's',
 't',
 'tʰ',
 'tʲ',
 'tʷ',
 't̪',
 'u',
 'v',
 'vʲ',
 'w',
 'y',
 'z',
 'æ',
 'ð',
 'ø',
 'ŋ',
 'œ',
 'ɐ',
 'ɑ',
 'ɑː',
 'ɒ',
 'ɒː',
 'ɔ',
 'ɖ',
 'ɗ',
 'ə',
 'ɛ',
 'ɜ',
 'ɜː',
 'ɟ',
 'ɠ',
 'ɡ',
 'ɪ',
 'ɫ',
 'ɫ̩',
 'ɯ',
 'ɲ',
 'ɳ',
 'ɹ',
 'ɾ',
 'ɾʲ',
 'ɾ̃',
 'ʃ',
 'ʉ',
 'ʉː',
 'ʊ',
 'ʌ',
 'ʎ',
 'ʒ',
 'θ'}

In [8]:
powsm_phones

{'ɜ̟',
 'cⁿ',
 'ɬ̪ʰˠ',
 'ʉ̯',
 't͡sʷˠʰ',
 'd̰ː',
 'n̪̻',
 'p͡ɸˤ',
 'ʉˠ',
 'b͡βʰʲ',
 'ɤ̠',
 'v̤ˠ',
 'ɫ̥ˤ',
 'ɢ',
 '˧ˤ',
 'ɒ̝',
 'ɟʷː',
 'ø̤',
 'ɖʰ',
 'ɡ͡bʷʰ',
 'ħʰˠ',
 'b͡dʷʰ',
 'ɪ̃ː',
 'd̪͡ðːʲ',
 'd͡ʒʲʷʰ',
 'ɱ̥ː',
 'z̰ʷ',
 't͡ʃʲʷ',
 'pʷ',
 'ʒːˤ',
 'ɞː',
 'ˀɠ',
 'ḭː',
 'ɺʲ',
 'tʷˀ',
 'ɦ̰ˠ',
 'β̤ː',
 'ʙ̰',
 'ɭ̞',
 'ʒːʲ',
 'p͡tʲʼ',
 't͡ɕʰᶣ',
 'ɾ̃ˤ',
 'ɨ̤',
 'z̟',
 't͡ɬʷʼ',
 'ɟʲʷʰ',
 'ɡ͡bˤʰ',
 'ǀˀ',
 'ʙ̤ː',
 'ʟːˤ',
 'ʀ̤̥',
 'ˀɮ',
 'ɻʲː',
 's̪ʷˠ',
 'ʃʷˤʰ',
 'ɗʷː',
 '́',
 'ɔˤ',
 'ʂʷː',
 'ˀn',
 'l̻',
 'ðˠ',
 'n̤ʲ',
 'd͡z̰ˠ',
 'd͡z̤ˠ',
 'θˠ',
 'ʏ̤ː',
 'ʒ̤ˤ',
 'ɯ̥ː',
 't͡ʃʲʷʰ',
 'ŋ̤ʷ',
 'b͡d̰̃',
 'd͡ʑʷ',
 'b͡vʲʷʰ',
 'kʼʷ',
 'ɹːˠ',
 'd̪͡ɮ̪̰ˤ',
 'ɮˤ',
 'd̪͡ðʷˤ',
 'ɳˤ',
 'ɻˤː',
 'd̪͡ð̰',
 'ʟʷˤ',
 'd͡ɮʰ',
 'ɣ̤ˤ',
 'ɟ̰ʷ',
 'd̪͡z̪ʰˠ',
 'ɐ̥ː',
 'ʒ̃ˤ',
 'œ̰ˠ',
 'ɻ̤',
 'ʛ̰ˠ',
 'q͡χʷˀ',
 'dʷˀ',
 'ʋ̰̃',
 'ɟ͡ʝ̃ː',
 'a',
 'ɡ͡b̃',
 'n',
 'χʰᶣ',
 'k͡xʷ',
 't̪ʰ',
 'ʐ̰ʲ',
 'ɫ̰̃',
 'ŋ̤ː',
 'fːʷ',
 'ʑ̤ˠ',
 'ʔʲ',
 't͡sːˠ',
 'ɺ̃ˤ',
 'ʛːˠ',
 'ʒːʷ',
 'ø̠',
 'l̩',
 'ʀ̰ʲ',
 'ɗˤ',
 'ɻːʲ',
 't̪ː',
 'ʋ̥ˤ',
 'ɳʲʷ',
 'ɱ̰ʲ'